In [ ]:
import os
import random
import json
import time
import pickle
import shutil
import numpy as np
import pandas as pd
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

#set seeds for reproductivity
GLOBAL_SEED = 
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")


def set_seed(seed=GLOBAL_SEED, deterministic=True):
    """Set Python, NumPy, and PyTorch seeds for repeatable training runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(GLOBAL_SEED)

In [ ]:
class SixHourlyFRDataset(Dataset):
    """Dataset for 6-hourly FR prediction with source-sink lagged features"""
    
    def __init__(self, 
                 data_path: str,
                 lookback_steps: int = 12,
                 forecast_steps: List[int] = [1, 2, 3, 4],
                 train_years: List[int] = None,
                 cache_dir: str = './cache_6hourly',
                 use_source_sink: bool = True,
                 source_lag_steps: List[int] = [1, 2],
                 source_bbox: Dict = None,
                 sink_bboxes: List[Dict] = None,  # Support multiple sink regions
                 aggregate_source: str = 'mean',
                 scaler: StandardScaler = None,  # Pass existing scaler
                 feature_cols: List[str] = None):  # Pass existing feature columns
        
        self.lookback_steps = lookback_steps
        self.forecast_steps = forecast_steps
        self.cache_dir = cache_dir
        self.use_source_sink = use_source_sink
        self.source_lag_steps = source_lag_steps if use_source_sink else []
        self.aggregate_source = aggregate_source
        
        # Define source region
        self.source_bbox = source_bbox or {
            "min_lat": 32.8, "max_lat": 37.1, 
            "min_lon": 106.2, "max_lon": 108.1
        }
        
        # Handle sink regions - can be None (all non-source), single dict, or list
        if sink_bboxes is None:
            self.sink_bboxes = None
            self.use_all_non_source_as_sink = True
        elif isinstance(sink_bboxes, dict):
            self.sink_bboxes = [sink_bboxes]
            self.use_all_non_source_as_sink = False
        else:
            self.sink_bboxes = sink_bboxes
            self.use_all_non_source_as_sink = False
        
        # Create cache directory
        os.makedirs(cache_dir, exist_ok=True)
        
        # Create cache file name
        cache_suffix = "_ss" if use_source_sink else "_baseline"
        if self.use_all_non_source_as_sink:
            cache_suffix += "_allsink"
        self.cache_file = os.path.join(cache_dir, f'dataset_cache{cache_suffix}.pkl')
        
        # Try to load from cache
        if os.path.exists(self.cache_file):
            print(f"Loading from cache: {self.cache_file}")
            try:
                with open(self.cache_file, 'rb') as f:
                    cache_data = pickle.load(f)
                    for key, value in cache_data.items():
                        setattr(self, key, value)
                print(f"Loaded {len(self.sequences)} sequences from cache")
                return
            except Exception as e:
                print(f"Cache loading failed: {e}. Rebuilding dataset...")
        
        # Build dataset from scratch
        print("Building 6-hourly dataset...")
        if use_source_sink:
            print(f"  Source region: {self.source_bbox}")
            if self.use_all_non_source_as_sink:
                print("  Sink region: All non-source data")
            else:
                print(f"  Sink regions: {len(self.sink_bboxes)} regions")
                
        start_time = time.time()
        
        # Load data
        df = pd.read_csv(data_path)
        print(f"Loaded {len(df)} rows")
        
        # Determine feature columns
        if feature_cols is not None:
            self.feature_cols = feature_cols
        else:
            # Auto-detect feature columns
            base_features = ['is_FR', 'tas', 'RH', 'WbT', 'WS', 'TP', 'DT']
            level_features = ['T975', 'T950', 'T900', 'T850', 'T800', 'T750', 'T700', 'T650', 'T500',
                              '975RH', '950RH', '900RH', '850RH', '800RH', '750RH', '700RH', '650RH', '500RH',
                              '975VV', '950VV', '900VV', '850VV', '800VV', '750VV', '700VV', '650VV', '500VV']
            
            self.feature_cols = []
            for col in base_features + level_features:
                if col in df.columns and col not in self.feature_cols:
                    self.feature_cols.append(col)
            
            # Ensure is_FR is first
            if 'is_FR' in self.feature_cols:
                self.feature_cols.remove('is_FR')
                self.feature_cols.insert(0, 'is_FR')
        
        print(f"Using {len(self.feature_cols)} features: {self.feature_cols[:5]}...")
        
        
        # Create datetime
        df['datetime'] = pd.to_datetime(
            df[['year', 'month', 'day', 'hour']].assign(minute=0, second=0)
        )
        
        # Filter winter months
        df = df[df['datetime'].dt.month.isin([12, 1, 2])]
        
        # Add winter season
        df['winter_season'] = df['datetime'].apply(
            lambda x: x.year if x.month <= 2 else x.year + 1
        )
        
        # Filter years if specified
        if train_years:
            df = df[df['winter_season'].isin(train_years)]
            print(f"Filtered to {len(df)} rows for years {train_years[0]}-{train_years[-1]}")
        
        # Sort by location and time
        df = df.sort_values(['latitude', 'longitude', 'datetime'])
        
        # Separate source and sink data
        source_mask = (
            (df['latitude'] >= self.source_bbox['min_lat']) & 
            (df['latitude'] <= self.source_bbox['max_lat']) &
            (df['longitude'] >= self.source_bbox['min_lon']) & 
            (df['longitude'] <= self.source_bbox['max_lon'])
        )
        
        self.source_df = df[source_mask].copy()
        
        # Get sink data
        if self.use_all_non_source_as_sink:
            df_sink = df[~source_mask].copy()
        else:
            sink_mask = pd.Series(False, index=df.index)
            for bbox in self.sink_bboxes:
                region_mask = (
                    (df['latitude'] >= bbox['min_lat']) & 
                    (df['latitude'] <= bbox['max_lat']) &
                    (df['longitude'] >= bbox['min_lon']) & 
                    (df['longitude'] <= bbox['max_lon'])
                )
                sink_mask = sink_mask | region_mask
            sink_mask = sink_mask & (~source_mask)
            df_sink = df[sink_mask].copy()
        
        print(f"Source region: {len(self.source_df)} rows")
        print(f"Sink region: {len(df_sink)} rows")
        
        if self.use_source_sink and len(self.source_df) == 0:
            raise ValueError("No data found in source region!")
        if len(df_sink) == 0:
            raise ValueError("No data found in sink region!")
        
        # Prepare source features if using source-sink
        if self.use_source_sink:
            self._prepare_source_features()
        
        # Normalize features
        print("Normalizing features...")
        norm_features = [col for col in self.feature_cols if col != 'is_FR']
        
        if scaler is not None:
            self.scaler = scaler
            print("Using provided scaler")
        else:
            self.scaler = StandardScaler()
            if self.use_source_sink:
                all_data = pd.concat([df_sink[norm_features], self.source_df[norm_features]])
                self.scaler.fit(all_data.values)
            else:
                self.scaler.fit(df_sink[norm_features].values)
            print("Fitted new scaler")
        
        # Transform data
        if norm_features:
            df_sink.loc[:, norm_features] = self.scaler.transform(df_sink[norm_features].values)
            if self.use_source_sink and len(self.source_df) > 0:
                self.source_df.loc[:, norm_features] = self.scaler.transform(
                    self.source_df[norm_features].values
                )
        
        # Create sequences
        print("Creating sequences...")
        self._create_sequences(df_sink)
        
        # Store metadata
        if len(self.sequences) > 0:
            self.num_features_total = self.sequences.shape[-1]
            if self.use_source_sink:
                print(f"Total features per timestep: {self.num_features_total} "
                      f"(original: {len(self.feature_cols)}, "
                      f"source: {self.num_features_total - len(self.feature_cols)})")
        else:
            self.num_features_total = len(self.feature_cols)
        
        print(f"\nDataset creation summary:")
        print(f"  Total sequences: {len(self.sequences)}")
        print(f"  Sequence shape: {self.sequences.shape if len(self.sequences) > 0 else 'No sequences'}")
        print(f"  Dataset built in {time.time() - start_time:.2f} seconds")
        
        # Analyze class distribution
        self._analyze_class_distribution()
        
        # Save to cache
        self._save_to_cache()
    
    def _prepare_source_features(self):
        """Prepare time-indexed source features"""
        print("Preparing source region features...")
        
        self.source_by_time = {}
        
        for time, time_data in self.source_df.groupby('datetime'):
            features = time_data[self.feature_cols].values
            
            if self.aggregate_source == 'mean':
                aggregated = np.mean(features, axis=0)
            elif self.aggregate_source == 'max':
                aggregated = np.max(features, axis=0)
            else:
                aggregated = np.mean(features, axis=0)
            
            self.source_by_time[time] = aggregated
        
        self.source_times = sorted(self.source_by_time.keys())
        print(f"Prepared {len(self.source_times)} source timesteps")
    
    def _get_lagged_source_features(self, current_time: pd.Timestamp) -> Optional[np.ndarray]:
        """Get source features at specified lags"""
        lagged_features = []
        
        for lag in self.source_lag_steps:
            lag_time = current_time - pd.Timedelta(hours=lag * 6)
            
            if lag_time in self.source_by_time:
                features = self.source_by_time[lag_time]
            else:
                nearest_time = self._find_nearest_time(lag_time)
                if nearest_time is None:
                    return None
                features = self.source_by_time[nearest_time]
            
            lagged_features.append(features)
        
        return np.concatenate(lagged_features)
    
    def _find_nearest_time(self, target_time: pd.Timestamp, max_diff_hours: int = 6) -> Optional[pd.Timestamp]:
        """Find nearest available time in source data"""
        if not self.source_times:
            return None
        
        idx = np.searchsorted([t.value for t in self.source_times], target_time.value)
        
        candidates = []
        if idx > 0:
            candidates.append(self.source_times[idx - 1])
        if idx < len(self.source_times):
            candidates.append(self.source_times[idx])
        
        if not candidates:
            return None
        
        min_diff = float('inf')
        nearest = None
        for candidate in candidates:
            diff = abs((candidate - target_time).total_seconds() / 3600)
            if diff < min_diff and diff <= max_diff_hours:
                min_diff = diff
                nearest = candidate
        
        return nearest
    
    def _create_sequences(self, df):
        """Create training sequences"""
        self.sequences = []
        self.targets = []
        self.metadata = []
        
        location_groups = df.groupby(['latitude', 'longitude'])
        print(f"Processing {len(location_groups)} locations...")
        
        for (lat, lon), location_data in tqdm(location_groups, desc="Creating sequences"):
            location_data = location_data.sort_values('datetime').reset_index(drop=True)
            
            if len(location_data) < self.lookback_steps + max(self.forecast_steps):
                continue
            
            features = location_data[self.feature_cols].values
            times = location_data['datetime'].values
            
            for i in range(self.lookback_steps, len(location_data) - max(self.forecast_steps)):
                lookback_features = features[i - self.lookback_steps:i]
                current_time = pd.to_datetime(times[i-1])
                
                if self.use_source_sink:
                    source_features = self._get_lagged_source_features(current_time)
                    if source_features is not None:
                        enhanced_features = np.concatenate([
                            lookback_features,
                            np.repeat(source_features[np.newaxis, :], self.lookback_steps, axis=0)
                        ], axis=1)
                    else:
                        continue
                else:
                    enhanced_features = lookback_features
                
                target_dict = {}
                valid_sequence = True
                
                for step in self.forecast_steps:
                    target_idx = i + step - 1
                    if target_idx < len(features):
                        target_dict[step] = features[target_idx, 0]  # is_FR value
                    else:
                        valid_sequence = False
                        break
                
                if valid_sequence:
                    self.sequences.append(enhanced_features)
                    self.targets.append(target_dict)
                    self.metadata.append({
                        'latitude': lat,
                        'longitude': lon,
                        'current_time': current_time,
                        'is_FR_current': features[i-1, 0]
                    })
        
        self.sequences = np.array(self.sequences, dtype=np.float32) if self.sequences else np.array([])
    
    def _analyze_class_distribution(self):
        """Analyze and print class distribution"""
        if len(self.targets) == 0:
            print("\nNo sequences to analyze")
            return
            
        print("\nClass distribution (is_FR events):")
        for step in self.forecast_steps:
            positive_count = sum(1 for t in self.targets if t[step] == 1)
            total_count = len(self.targets)
            positive_ratio = positive_count / total_count if total_count > 0 else 0
            
            print(f"  {step*6}h ahead: {positive_count}/{total_count} positive "
                  f"({positive_ratio*100:.2f}%)")
    
    def _save_to_cache(self):
        """Save dataset to cache"""
        if len(self.sequences) > 0:
            cache_data = {
                'sequences': self.sequences,
                'targets': self.targets,
                'metadata': self.metadata,
                'feature_cols': self.feature_cols,
                'lookback_steps': self.lookback_steps,
                'forecast_steps': self.forecast_steps,
                'scaler': self.scaler,
                'use_source_sink': self.use_source_sink,
                'source_lag_steps': self.source_lag_steps,
                'num_features_total': self.num_features_total,
                'source_bbox': self.source_bbox,
                'sink_bboxes': self.sink_bboxes,
                'use_all_non_source_as_sink': self.use_all_non_source_as_sink
            }
            
            if hasattr(self, 'source_by_time'):
                cache_data['source_by_time'] = self.source_by_time
                cache_data['source_times'] = self.source_times
            
            try:
                with open(self.cache_file, 'wb') as f:
                    pickle.dump(cache_data, f)
                print(f"Saved to cache: {self.cache_file}")
            except Exception as e:
                print(f"Failed to save cache: {e}")
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return {
            'sequence': torch.tensor(self.sequences[idx], dtype=torch.float32),
            'targets': {
                step: torch.tensor(self.targets[idx][step], dtype=torch.float32)
                for step in self.forecast_steps
            }
        }

In [ ]:
# MODEL ARCHITECTURE
class ResNetBlock1D(nn.Module):
    """Standard 1D ResNet Block"""
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = downsample
        
    def forward(self, x):
        identity = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.downsample is not None:
            identity = self.downsample(x)
            
        out += identity
        out = self.relu(out)
        
        return out


class ResNetLSTMTransformer(nn.Module):
    """
    Hybrid Architecture: ResNet + Bi-LSTM + Multi-Head Self-Attention 
    Includes Gated Skip Connections
    """
    
    def __init__(self,
                 num_features: int,
                 lookback_steps: int,
                 hidden_dim: int = 128,
                 lstm_layers: int = 4,
                 resnet_layers: List[int] = [4, 4, 4],
                 base_channels: int = 32,
                 dropout: float = 0.2,
                 forecast_steps: List[int] = [1, 2, 3, 4],
                 num_heads: int = 4,  
                 use_skip_connection: bool = True):
        super().__init__()
        
        self.num_features = num_features
        self.lookback_steps = lookback_steps
        self.hidden_dim = hidden_dim
        self.forecast_steps = forecast_steps
        self.use_skip_connection = use_skip_connection
        
        # ResNet Block 
        self.conv1 = nn.Conv1d(num_features, base_channels, kernel_size=7, 
                               stride=1, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(base_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        
        # Initialize ResNet layers
        self.layer1 = self._make_layer(base_channels, base_channels, resnet_layers[0])
        self.layer2 = self._make_layer(base_channels, base_channels * 2, resnet_layers[1], stride=2)
        self.layer3 = self._make_layer(base_channels * 2, base_channels * 4, resnet_layers[2], stride=2)
        
        pooled_steps = max(int(lookback_steps * 0.75), min(8, lookback_steps))
        self.global_pool = nn.AdaptiveAvgPool1d(pooled_steps)
        
        # LSTM Block
        lstm_input_size = base_channels * 4
        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0,
            bidirectional=True
        )
        
        # Dimensions check for Multi-Head Attention
        lstm_output_dim = hidden_dim * 2  # *2 for Bidirectional
        if lstm_output_dim % num_heads != 0:
            raise ValueError(f"LSTM output dim ({lstm_output_dim}) must be divisible by num_heads ({num_heads})")

        self.pre_lstm_norm = nn.LayerNorm(lstm_input_size)
        self.post_lstm_norm = nn.LayerNorm(lstm_output_dim)
        
        # --- Multi-Head Self-Attention ---
        self.mha = nn.MultiheadAttention(
            embed_dim=lstm_output_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.post_mha_norm = nn.LayerNorm(lstm_output_dim)
        
        # --- Gated Skip Connection ---
        if self.use_skip_connection:
            self.skip_projection = nn.Linear(lstm_output_dim, lstm_output_dim)
            self.attention_gate = nn.Sequential(
                nn.Linear(lstm_output_dim * 2, lstm_output_dim),
                nn.ReLU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(lstm_output_dim, lstm_output_dim),
                nn.Sigmoid()
            )
        
        # --- Prediction Heads ---
        self.prediction_heads = nn.ModuleDict()
        for step in forecast_steps:
            self.prediction_heads[str(step)] = nn.Sequential(
                nn.LayerNorm(lstm_output_dim),
                nn.Linear(lstm_output_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1)
            )
            
        self.position_encoding = nn.Parameter(
            torch.randn(1, pooled_steps, lstm_input_size) * 0.1
        )

    def _make_layer(self, in_channels, out_channels, num_blocks, stride=1):
        """Create a ResNet layer (FIXED: Now contains actual logic)"""
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        
        layers = []
        layers.append(ResNetBlock1D(in_channels, out_channels, stride=stride, downsample=downsample))
        
        for _ in range(1, num_blocks):
            layers.append(ResNetBlock1D(out_channels, out_channels))
        
        return nn.Sequential(*layers)

    def forward(self, x, return_attention=False):
        # 1. ResNet Feature Extraction
        x = x.transpose(1, 2) # [Batch, Feat, Time]
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x)
        
        # 2. Prepare for Sequence Modeling
        x = x.transpose(1, 2) # [Batch, Time, Feat]
        x = x + self.position_encoding[:, :x.size(1), :]
        x = self.pre_lstm_norm(x)
        
        # 3. LSTM
        lstm_out, _ = self.lstm(x)
        lstm_out = self.post_lstm_norm(lstm_out)
        
        # 4. Multi-Head Attention
        mha_out, attn_weights = self.mha(lstm_out, lstm_out, lstm_out)
        mha_out = self.post_mha_norm(mha_out + lstm_out)
        
        # 5. Pooling
        attended_output = mha_out.mean(dim=1) 
        
        # 6. Gated Skip Connection
        if self.use_skip_connection:
            skip_output = lstm_out.mean(dim=1)
            skip_output = self.skip_projection(skip_output)
            
            combined = torch.cat([attended_output, skip_output], dim=1)
            gate = self.attention_gate(combined)
            
            final_output = gate * attended_output + (1 - gate) * skip_output
        else:
            final_output = attended_output
            
        # 7. Prediction
        predictions = {}
        for step in self.forecast_steps:
            predictions[step] = self.prediction_heads[str(step)](final_output)
            
        if return_attention:
            return predictions, attn_weights
            
        return predictions

class SourceEnhancedTrainer:
    """
    Trainer for the source-enhanced ResNet-LSTM-Transformer model.

    By default, only the source-enhanced model is trained. Set
    train_baseline=True to train a no-source ablation for comparison. No
    fusion model or fusion weights are computed.
    """

    forecast_steps = [1, 2, 3, 4]
    horizons = [6, 12, 18, 24]
    metric_names = ['f1', 'ts', 'precision', 'recall', 'accuracy', 'bias', 'hss']

    def __init__(self,
                 data_path: str,
                 train_years: List[int],
                 val_years: List[int],
                 model_config: Dict = None,
                 training_config: Dict = None,
                 device: str = None,
                 source_lag_steps: List[int] = [2],
                 source_bbox: Dict = None,
                 sink_bboxes: List[Dict] = None,
                 output_dir: str = './models',
                 train_baseline: bool = False,
                 seed: int = GLOBAL_SEED,
                 deterministic: bool = True):

        self.data_path = data_path
        self.train_years = train_years
        self.val_years = val_years
        self.source_lag_steps = source_lag_steps
        self.source_bbox = source_bbox
        self.sink_bboxes = sink_bboxes
        self.output_dir = output_dir
        self.train_baseline = train_baseline
        self.seed = seed
        self.deterministic = deterministic

        os.makedirs(output_dir, exist_ok=True)

        set_seed(self.seed, self.deterministic)
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        print(f"Using seed: {self.seed} (deterministic={self.deterministic})")

        self.model_config = model_config or {
            'hidden_dim': 128,
            'lstm_layers': 3,
            'resnet_layers': [3, 4, 5],
            'base_channels': 32,
            'dropout': 0.2,
            'use_skip_connection': True,
            'num_heads': 4,
            'lookback_steps': 12
        }

        self.training_config = training_config or {
            'batch_size': 64,
            'learning_rate': 0.0001,
            'num_epochs': 20,
            'early_stopping_patience': 5,
            'pos_weight': 1.0
        }

        self.results = {
            'source': {
                'history': None,
                'best_epoch': None,        # 1-based epoch number for reporting
                'best_epoch_index': -1,    # 0-based index for history lookup
                'best_score': None
            }
        }
        if self.train_baseline:
            self.results['baseline'] = {
                'history': None,
                'best_epoch': None,
                'best_epoch_index': -1,
                'best_score': None
            }

    def _model_seed(self, model_name):
        offsets = {
            'baseline': 0,
            'source': 1000
        }
        return self.seed + offsets.get(model_name, 0)

    def _make_generator(self, model_name):
        generator = torch.Generator()
        generator.manual_seed(self._model_seed(model_name))
        return generator

    def train_baseline_model(self):
        """Train an optional no-source baseline model for ablation."""
        set_seed(self._model_seed('baseline'), self.deterministic)
        print("\n" + "="*70)
        print("TRAINING BASELINE MODEL (OPTIONAL ABLATION)")
        print("="*70)

        cache_dir = './cache_6hourly/baseline_train'
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir)

        self.train_dataset_baseline = SixHourlyFRDataset(
            data_path=self.data_path,
            train_years=self.train_years,
            cache_dir=cache_dir,
            use_source_sink=False,
            source_bbox=self.source_bbox,
            sink_bboxes=self.sink_bboxes,
            lookback_steps=self.model_config['lookback_steps']
        )

        self.baseline_scaler = self.train_dataset_baseline.scaler
        self.baseline_feature_cols = self.train_dataset_baseline.feature_cols

        val_cache_dir = './cache_6hourly/baseline_val'
        if os.path.exists(val_cache_dir):
            shutil.rmtree(val_cache_dir)

        self.val_dataset_baseline = SixHourlyFRDataset(
            data_path=self.data_path,
            train_years=self.val_years,
            cache_dir=val_cache_dir,
            use_source_sink=False,
            source_bbox=self.source_bbox,
            sink_bboxes=self.sink_bboxes,
            scaler=self.baseline_scaler,
            feature_cols=self.baseline_feature_cols,
            lookback_steps=self.model_config['lookback_steps']
        )

        print(f"\nBaseline dataset: Train={len(self.train_dataset_baseline)}, Val={len(self.val_dataset_baseline)}")

        train_loader = DataLoader(
            self.train_dataset_baseline,
            batch_size=self.training_config['batch_size'],
            shuffle=True,
            num_workers=0,
            drop_last=True,
            generator=self._make_generator('baseline'),
            worker_init_fn=seed_worker
        )

        val_loader = DataLoader(
            self.val_dataset_baseline,
            batch_size=self.training_config['batch_size'],
            shuffle=False,
            num_workers=0
        )

        sample = self.train_dataset_baseline[0]
        num_features = sample['sequence'].shape[1]
        lookback_steps = sample['sequence'].shape[0]

        print(f"Baseline model input: {lookback_steps} timesteps x {num_features} features")

        self.baseline_model = ResNetLSTMTransformer(
            num_features=num_features,
            lookback_steps=lookback_steps,
            forecast_steps=self.forecast_steps,
            hidden_dim=self.model_config['hidden_dim'],
            lstm_layers=self.model_config['lstm_layers'],
            resnet_layers=self.model_config['resnet_layers'],
            base_channels=self.model_config['base_channels'],
            dropout=self.model_config['dropout'],
            num_heads=self.model_config.get('num_heads', 4),
            use_skip_connection=self.model_config['use_skip_connection']
        ).to(self.device)

        print(f"Baseline model parameters: {sum(p.numel() for p in self.baseline_model.parameters()):,}")

        history = self._train_model(
            self.baseline_model, train_loader, val_loader, "baseline"
        )

        self.results['baseline']['history'] = history

        model_path = os.path.join(self.output_dir, 'baseline_model.pth')
        torch.save(self.baseline_model.state_dict(), model_path)
        print(f"[OK] Baseline model saved to {model_path}")

        return history

    def train_source_model(self):
        """Train the source-enhanced model."""
        set_seed(self._model_seed('source'), self.deterministic)
        print("\n" + "="*70)
        print("TRAINING SOURCE-ENHANCED MODEL")
        print("="*70)

        cache_dir = './cache_6hourly/source_train'
        if os.path.exists(cache_dir):
            shutil.rmtree(cache_dir)

        self.train_dataset_source = SixHourlyFRDataset(
            data_path=self.data_path,
            train_years=self.train_years,
            cache_dir=cache_dir,
            use_source_sink=True,
            source_lag_steps=self.source_lag_steps,
            source_bbox=self.source_bbox,
            sink_bboxes=self.sink_bboxes,
            lookback_steps=self.model_config['lookback_steps']
        )

        self.scaler = self.train_dataset_source.scaler
        self.feature_cols = self.train_dataset_source.feature_cols

        val_cache_dir = './cache_6hourly/source_val'
        if os.path.exists(val_cache_dir):
            shutil.rmtree(val_cache_dir)

        self.val_dataset_source = SixHourlyFRDataset(
            data_path=self.data_path,
            train_years=self.val_years,
            cache_dir=val_cache_dir,
            use_source_sink=True,
            source_lag_steps=self.source_lag_steps,
            source_bbox=self.source_bbox,
            sink_bboxes=self.sink_bboxes,
            scaler=self.scaler,
            feature_cols=self.feature_cols,
            lookback_steps=self.model_config['lookback_steps']
        )

        print(f"\nSource dataset: Train={len(self.train_dataset_source)}, Val={len(self.val_dataset_source)}")

        train_loader = DataLoader(
            self.train_dataset_source,
            batch_size=self.training_config['batch_size'],
            shuffle=True,
            num_workers=0,
            drop_last=True,
            generator=self._make_generator('source'),
            worker_init_fn=seed_worker
        )

        val_loader = DataLoader(
            self.val_dataset_source,
            batch_size=self.training_config['batch_size'],
            shuffle=False,
            num_workers=0
        )

        sample = self.train_dataset_source[0]
        num_features = sample['sequence'].shape[1]
        lookback_steps = sample['sequence'].shape[0]

        print(f"Source model input: {lookback_steps} timesteps x {num_features} features")
        print(f"  (Original: {len(self.feature_cols)}, Source: {num_features - len(self.feature_cols)})")

        self.source_model = ResNetLSTMTransformer(
            num_features=num_features,
            lookback_steps=lookback_steps,
            forecast_steps=self.forecast_steps,
            hidden_dim=self.model_config['hidden_dim'],
            lstm_layers=self.model_config['lstm_layers'],
            resnet_layers=self.model_config['resnet_layers'],
            base_channels=self.model_config['base_channels'],
            dropout=self.model_config['dropout'],
            num_heads=self.model_config.get('num_heads', 4),
            use_skip_connection=self.model_config['use_skip_connection']
        ).to(self.device)

        print(f"Source model parameters: {sum(p.numel() for p in self.source_model.parameters()):,}")

        history = self._train_model(
            self.source_model, train_loader, val_loader, "source"
        )

        self.results['source']['history'] = history

        model_path = os.path.join(self.output_dir, 'source_model.pth')
        torch.save(self.source_model.state_dict(), model_path)
        print(f"[OK] Source model saved to {model_path}")

        return history

    def _train_model(self, model, train_loader, val_loader, model_name):
        """Train a single model and keep the best checkpoint by average TS."""

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=self.training_config['learning_rate'],
            weight_decay=1e-5
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(self.training_config['pos_weight']).to(self.device)
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=3, factor=0.5
        )

        history = {
            'train_loss': [],
            'val_loss': [],
            'val_metrics': {
                h: {metric: [] for metric in self.metric_names}
                for h in self.horizons
            }
        }

        best_score = -np.inf
        best_epoch = -1
        patience_counter = 0

        for epoch in range(self.training_config['num_epochs']):
            model.train()
            total_train_loss = 0

            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
                sequences = batch['sequence'].to(self.device)
                predictions = model(sequences)

                loss = 0
                for step in self.forecast_steps:
                    targets = batch['targets'][step].to(self.device).unsqueeze(1)
                    loss += criterion(predictions[step], targets)

                loss = loss / len(self.forecast_steps)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                total_train_loss += loss.item()

            train_loss = total_train_loss / len(train_loader)

            model.eval()
            total_val_loss = 0
            metrics = {step: {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0} for step in self.forecast_steps}

            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Validating"):
                    sequences = batch['sequence'].to(self.device)
                    predictions = model(sequences)

                    loss = 0
                    for step in self.forecast_steps:
                        targets = batch['targets'][step].to(self.device).unsqueeze(1)
                        loss += criterion(predictions[step], targets)

                        probs = torch.sigmoid(predictions[step])
                        preds = (probs > 0.5).float()

                        metrics[step]['tp'] += ((preds == 1) & (targets == 1)).sum().item()
                        metrics[step]['fp'] += ((preds == 1) & (targets == 0)).sum().item()
                        metrics[step]['tn'] += ((preds == 0) & (targets == 0)).sum().item()
                        metrics[step]['fn'] += ((preds == 0) & (targets == 1)).sum().item()

                    total_val_loss += (loss / len(self.forecast_steps)).item()

            val_loss = total_val_loss / len(val_loader)

            val_results = {}
            for step, h in zip(self.forecast_steps, self.horizons):
                m = metrics[step]
                precision = m['tp'] / (m['tp'] + m['fp'] + 1e-8)
                recall = m['tp'] / (m['tp'] + m['fn'] + 1e-8)
                f1 = 2 * precision * recall / (precision + recall + 1e-8)
                accuracy = (m['tp'] + m['tn']) / (m['tp'] + m['tn'] + m['fp'] + m['fn'] + 1e-8)
                ts = m['tp'] / (m['tp'] + m['fn'] + m['fp'] + 1e-8)
                bias = (m['tp'] + m['fp']) / (m['tp'] + m['fn'] + 1e-8)

                numerator = 2 * (m['tp'] * m['tn'] - m['fp'] * m['fn'])
                denominator = ((m['tp'] + m['fn']) * (m['fn'] + m['tn']) +
                               (m['tp'] + m['fp']) * (m['fp'] + m['tn']))
                hss = numerator / (denominator + 1e-8)

                val_results[h] = {
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'accuracy': accuracy,
                    'ts': ts,
                    'bias': bias,
                    'hss': hss
                }

            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)

            for h in self.horizons:
                for metric in self.metric_names:
                    history['val_metrics'][h][metric].append(val_results[h][metric])

            scheduler.step(val_loss)

            print(f"\nEpoch {epoch+1}/{self.training_config['num_epochs']} ({model_name})")
            print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")

            print("\nValidation Metrics:")
            for h in self.horizons:
                m = val_results[h]
                print(f"  {h}h: F1={m['f1']:.3f}, Recall={m['recall']:.3f}, Precision={m['precision']:.3f}, TS={m['ts']:.3f}, Bias={m['bias']:.3f}, HSS={m['hss']:.3f}")

            avg_ts = np.mean([val_results[h]['ts'] for h in self.horizons])
            current_score = avg_ts

            if current_score > best_score:
                best_score = current_score
                best_epoch = epoch
                patience_counter = 0

                best_model_path = os.path.join(self.output_dir, f'best_{model_name}_model.pth')
                torch.save(model.state_dict(), best_model_path)
                print(f"[OK] Saved best {model_name} model (avg TS: {current_score:.3f})")
            else:
                patience_counter += 1

            if patience_counter >= self.training_config['early_stopping_patience']:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        print(f"\n{model_name.capitalize()} training complete! Best epoch: {best_epoch+1}")

        best_model_path = os.path.join(self.output_dir, f'best_{model_name}_model.pth')
        if os.path.exists(best_model_path):
            model.load_state_dict(torch.load(best_model_path, map_location=self.device))

        self.results[model_name]['best_epoch_index'] = best_epoch
        self.results[model_name]['best_epoch'] = best_epoch + 1 if best_epoch >= 0 else None
        self.results[model_name]['best_score'] = float(best_score) if np.isfinite(best_score) else None

        return history

    def _metrics_at_best_epoch(self, model_name):
        result = self.results[model_name]
        history = result['history']
        if history is None:
            return None

        best_epoch_index = result.get('best_epoch_index', -1)
        if best_epoch_index < 0:
            best_epoch_index = len(history['val_loss']) - 1

        return {
            h: {
                metric: history['val_metrics'][h][metric][best_epoch_index]
                for metric in self.metric_names
            }
            for h in self.horizons
        }

    def _metrics_by_epoch(self, model_name):
        """Return validation metrics in the same epoch order as the training log."""
        history = self.results[model_name]['history']
        if history is None:
            return None

        epoch_rows = []
        for epoch_index, (train_loss, val_loss) in enumerate(zip(history['train_loss'], history['val_loss'])):
            metrics = {
                h: {
                    metric: history['val_metrics'][h][metric][epoch_index]
                    for metric in self.metric_names
                }
                for h in self.horizons
            }
            epoch_rows.append({
                'epoch': epoch_index + 1,
                'epoch_index': epoch_index,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'mean_validation_ts': float(np.mean([metrics[h]['ts'] for h in self.horizons])),
                'metrics': metrics
            })

        return epoch_rows

    def save_all(self):
        """Save model artifacts, source scaler, and training results."""
        print("\n" + "="*70)
        print("SAVING OUTPUTS")
        print("="*70)

        if not hasattr(self, 'scaler') or not hasattr(self, 'feature_cols'):
            raise RuntimeError("Source model must be trained before saving artifacts.")

        scaler_path = os.path.join(self.output_dir, 'scaler.pkl')
        scaler_data = {
            'scaler': self.scaler,
            'feature_cols': self.feature_cols,
            'source_bbox': self.source_bbox,
            'sink_bboxes': self.sink_bboxes,
            'source_lag_steps': self.source_lag_steps,
            'model_config': self.model_config,
            'training_config': self.training_config,
            'train_baseline': self.train_baseline,
            'seed': self.seed,
            'deterministic': self.deterministic,
            'use_source_sink': True
        }
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler_data, f)
        print(f"[OK] Source scaler saved to {scaler_path}")

        results_path = os.path.join(self.output_dir, 'training_results.json')

        def convert_numpy(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            if isinstance(obj, np.floating):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, dict):
                return {k: convert_numpy(v) for k, v in obj.items()}
            if isinstance(obj, list):
                return [convert_numpy(i) for i in obj]
            return obj

        results_to_save = {
            'run_metadata': {
                'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
                'selection_metric': 'mean_validation_ts',
                'selection_metric_description': 'Mean TS over 6h, 12h, 18h, and 24h horizons.'
            },
            'config': {
                'source_bbox': self.source_bbox,
                'sink_bboxes': self.sink_bboxes,
                'source_lag_steps': self.source_lag_steps,
                'train_years': self.train_years,
                'val_years': self.val_years,
                'model_config': self.model_config,
                'training_config': self.training_config,
                'train_baseline': self.train_baseline,
                'seed': self.seed,
                'deterministic': self.deterministic
            },
            'source': {
                'best_epoch': self.results['source']['best_epoch'],
                'best_epoch_index': self.results['source']['best_epoch_index'],
                'selection_metric': 'mean_validation_ts',
                'best_score': self.results['source']['best_score'],
                'best_metrics': self._metrics_at_best_epoch('source'),
                'metrics_by_epoch': self._metrics_by_epoch('source'),
                'history': self.results['source']['history']
            }
        }

        if self.train_baseline and self.results.get('baseline', {}).get('history') is not None:
            results_to_save['baseline'] = {
                'best_epoch': self.results['baseline']['best_epoch'],
                'best_epoch_index': self.results['baseline']['best_epoch_index'],
                'selection_metric': 'mean_validation_ts',
                'best_score': self.results['baseline']['best_score'],
                'best_metrics': self._metrics_at_best_epoch('baseline'),
                'metrics_by_epoch': self._metrics_by_epoch('baseline'),
                'history': self.results['baseline']['history']
            }

        results_to_save = convert_numpy(results_to_save)

        with open(results_path, 'w') as f:
            json.dump(results_to_save, f, indent=2)
        print(f"[OK] Training results saved to {results_path}")

        print(f"\n[OK] Outputs saved to {self.output_dir}/")
        print("  - source_model.pth")
        print("  - best_source_model.pth")
        if self.train_baseline and self.results.get('baseline', {}).get('history') is not None:
            print("  - baseline_model.pth")
            print("  - best_baseline_model.pth")
        print("  - scaler.pkl")
        print("  - training_results.json")

    def train(self):
        """Run the training pipeline without model fusion."""
        print("\n" + "="*70)
        print("SOURCE-ENHANCED TRAINING PIPELINE")
        print("="*70)
        print(f"Data path: {self.data_path}")
        print(f"Train years: {self.train_years[0]}-{self.train_years[-1]}")
        print(f"Val years: {self.val_years[0]}-{self.val_years[-1]}")
        print(f"Source bbox: {self.source_bbox}")
        print(f"Source lag steps: {[lag*6 for lag in self.source_lag_steps]}h")
        print(f"Train baseline ablation: {self.train_baseline}")
        print(f"Seed: {self.seed} (deterministic={self.deterministic})")

        if self.train_baseline:
            self.train_baseline_model()

        self.train_source_model()

        recommendations = self.compare_models()
        self.save_all()

        print("\n" + "="*70)
        print("TRAINING COMPLETE!")
        print("="*70)

        if self.train_baseline:
            print("\nBEST MODEL BY HORIZON:")
            for h in ['6h', '12h', '18h', '24h']:
                rec = recommendations[h]
                print(f"  {h}: {rec['best_model'].capitalize()} (F1={rec['f1']:.3f})")
        else:
            print("\nSource-enhanced model trained as the deployment model.")

        return self.results

In [ ]:
if __name__ == "__main__":
    # Configuration - UPDATE THESE PATHS
    DATA_PATH = 
    OUTPUT_DIR = 

    # Years
    TRAIN_YEARS = list(range(2000, 2016))  # 2000-2015
    VAL_YEARS = list(range(2016, 2020))     # 2016-2019

    # Source region (upstream weather)
    SOURCE_BBOX = {
        "min_lat": 29.9, "max_lat": 32.4,
        "min_lon": 110.8, "max_lon": 112.2,
    }

    # Sink regions (can be None for all non-source, or list of bboxes)
    SINK_BBOXES = None  # Use all non-source data

    # Train the no-source baseline only when you need an ablation comparison.
    TRAIN_BASELINE = False

    # Model configuration
    MODEL_CONFIG = {
        'hidden_dim': 128,
        'lstm_layers': 3,
        'resnet_layers': [3, 4, 5],
        'base_channels': 32,
        'dropout': 0.20,
        'use_skip_connection': True,
        'lookback_steps': 12
    }

    # Training configuration
    TRAINING_CONFIG = {
        'batch_size': 64,
        'learning_rate': 0.0001,
        'num_epochs': 20,
        'early_stopping_patience': 2,
        'pos_weight': 1.0
    }

    # Source lag steps (in 6-hour intervals)
    SOURCE_LAG_STEPS = [1, 2, 3, 4]  # 6h, 12h lag

    # Reproducibility
    SEED = GLOBAL_SEED
    DETERMINISTIC = False

    trainer = SourceEnhancedTrainer(
        data_path=DATA_PATH,
        train_years=TRAIN_YEARS,
        val_years=VAL_YEARS,
        model_config=MODEL_CONFIG,
        training_config=TRAINING_CONFIG,
        source_lag_steps=SOURCE_LAG_STEPS,
        source_bbox=SOURCE_BBOX,
        sink_bboxes=SINK_BBOXES,
        output_dir=OUTPUT_DIR,
        train_baseline=TRAIN_BASELINE,
        seed=SEED,
        deterministic=DETERMINISTIC
    )

    results = trainer.train()